In [0]:
config = {
    "source_table": "bronze.employee",
    "target_table": "gold.employee_dim",
    "business_keys": ["EmpID"],
    "compare_columns": [
        "Name",
        "Department",
        "Salary"
    ]
}

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp,
    current_date
)

def scd_type2_load(config):

    source_table = config["source_table"]
    target_table = config["target_table"]
    business_keys = config["business_keys"]
    compare_columns = config["compare_columns"]

    # --------------------------
    # Read Source & Target
    # --------------------------
    source_df = spark.table(source_table)
    target_df = spark.table(target_table)

    # --------------------------
    # Generate Hash
    # --------------------------
    source_df = source_df.withColumn(
        "HashValue",
        sha2(concat_ws("||", *compare_columns), 256)
    )

    target_df = target_df.withColumn(
        "HashValue",
        sha2(concat_ws("||", *compare_columns), 256)
    )

    # --------------------------
    # Build Join Condition
    # --------------------------
    join_condition = [
        source_df[k] == target_df[k]
        for k in business_keys
    ]

    # --------------------------
    # Detect Changed/New Records
    # --------------------------
    changed_df = (
        source_df.alias("src")
        .join(
            target_df.filter(col("IsCurrent") == True).alias("tgt"),
            join_condition,
            "left"
        )
        .filter(
            (col("tgt.HashValue").isNull()) |
            (col("src.HashValue") != col("tgt.HashValue"))
        )
    )

    changed_df.createOrReplaceTempView("changed_records")

    # --------------------------
    # Expire Old Records
    # --------------------------
    key_condition = " AND ".join(
        [f"t.{k}=c.{k}" for k in business_keys]
    )

    spark.sql(f"""
        MERGE INTO {target_table} t
        USING changed_records c
        ON {key_condition}
        AND t.IsCurrent = true

        WHEN MATCHED THEN
        UPDATE SET
            t.EndDate = current_date(),
            t.IsCurrent = false,
            t.UpdatedTimestamp = current_timestamp()
    """)

    # --------------------------
    # Insert New Version
    # --------------------------
    new_df = (
        changed_df
        .drop("HashValue")
        .withColumn("StartDate", current_date())
        .withColumn("EndDate", lit(None).cast("date"))
        .withColumn("IsCurrent", lit(True))
        .withColumn("CreatedTimestamp", current_timestamp())
        .withColumn("UpdatedTimestamp", current_timestamp())
    )

    new_df.write \
        .mode("append") \
        .saveAsTable(target_table)

    print("SCD Type 2 Load Completed Successfully")

In [0]:
config = {

    "source_table":"bronze.employee",

    "target_table":"gold.employee_dim",

    "business_keys":["EmpID"],

    "compare_columns":[
        "Name",
        "Department",
        "Salary"
    ]

}

scd_type2_load(config)

In [0]:
%sql
CREATE TABLE gold.employee_dim
(
SurrogateKey BIGINT GENERATED ALWAYS AS IDENTITY,

EmpID INT,

Name STRING,

Department STRING,

Salary DOUBLE,

StartDate DATE,

EndDate DATE,

IsCurrent BOOLEAN,

CreatedTimestamp TIMESTAMP,

UpdatedTimestamp TIMESTAMP
)
USING DELTA;